# SCRUM-15 bounded exploratory global LightGBM forecasting evidence

This notebook presents an executed first global LightGBM regression experiment for Favorita. One shared model learns across multiple stores, items, historical origins, and direct forecast horizons 1 through 16.

The evidence is intentionally bounded and exploratory; it is organized for reproducible engineering and MSc review.

## 1. Experiment scope and bounded population

Fold 8 uses forecast origin `2017-06-30` and validation dates `2017-07-01` through `2017-07-16`. Training uses nine deterministic origins from `2017-02-28` through `2017-06-14`; their labels end at the Fold 8 origin.

Stores are exactly 1 through 5. For each store, eligible item identifiers are intersected across the earliest bounded training origin and the Fold 8 origin, sorted, and the lowest 100 retained. This is a bounded experiment and does not represent full Favorita training.

## 2. Locked forecast contract and deterministic setup

The experiment preserves signed `unit_sales`, end-of-day forecast origin `t`, exact horizons 1 through 16, `forecast_date = t+h`, direct horizon-aware prediction, no recursive feedback, no random split, and origin-bounded historical features.

LightGBM 4.7.0 is used from the active repository environment through its native `Dataset` and `train` APIs. The setup below fixes the bounded stores, training origins, Fold 8 dates, and source-read limits.

In [1]:
from pathlib import Path
from time import perf_counter

import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from IPython.display import display

from pipelines.evaluation.favorita_temporal_validation import (
    APPROVED_FOLDS,
    FINAL_HOLDOUT,
    FORECAST_HORIZONS,
    validate_forecast_date_horizon,
)
from pipelines.features.favorita_model_ready import (
    HOLIDAY_FEATURE_COLUMNS,
    INFERENCE_OUTPUT_COLUMNS,
    MODEL_FEATURE_COLUMNS,
    SOURCE_READ_COLUMNS,
    build_feature_rows_for_origin,
)

SOURCE_PATH = Path("data/processed/favorita_cleaned/favorita_cleaned.parquet")
STORES = (1, 2, 3, 4, 5)
ITEMS_PER_STORE = 100
TRAINING_ORIGINS = tuple(pd.Timestamp(value) for value in (
    "2017-02-28", "2017-03-14", "2017-03-28",
    "2017-04-11", "2017-04-25", "2017-05-09",
    "2017-05-23", "2017-06-06", "2017-06-14",
))
FOLD_8 = APPROVED_FOLDS[-1]
VALIDATION_ORIGIN = pd.Timestamp(FOLD_8.forecast_origin)
VALIDATION_START = pd.Timestamp(FOLD_8.validation_start)
VALIDATION_END = pd.Timestamp(FOLD_8.validation_end)
READ_START = TRAINING_ORIGINS[0] - pd.Timedelta(days=28)
READ_END = VALIDATION_END

assert lgb.__version__ == "4.7.0"
assert tuple(FORECAST_HORIZONS) == tuple(range(1, 17))
assert VALIDATION_ORIGIN == pd.Timestamp("2017-06-30")
assert VALIDATION_START == pd.Timestamp("2017-07-01")
assert VALIDATION_END == pd.Timestamp("2017-07-16")
assert READ_END < pd.Timestamp(FINAL_HOLDOUT.holdout_start)

print({
    "lightgbm_version": lgb.__version__,
    "stores": STORES,
    "training_origins": tuple(origin.date() for origin in TRAINING_ORIGINS),
    "validation_origin": VALIDATION_ORIGIN.date(),
    "validation_dates": (VALIDATION_START.date(), VALIDATION_END.date()),
})

{'lightgbm_version': '4.7.0', 'stores': (1, 2, 3, 4, 5), 'training_origins': (datetime.date(2017, 2, 28), datetime.date(2017, 3, 14), datetime.date(2017, 3, 28), datetime.date(2017, 4, 11), datetime.date(2017, 4, 25), datetime.date(2017, 5, 9), datetime.date(2017, 5, 23), datetime.date(2017, 6, 6), datetime.date(2017, 6, 14)), 'validation_origin': datetime.date(2017, 6, 30), 'validation_dates': (datetime.date(2017, 7, 1), datetime.date(2017, 7, 16))}


### Bounded population selection evidence

This table shows deterministic item eligibility and selection for each store. `store_nbr` is the store identifier; `earliest_origin_items` and `fold_origin_items` count items observed or eligible at the two selection dates; `intersection_items` counts items available across both required periods; and `selected_items` is the number retained for this bounded experiment.

In [2]:
assert SOURCE_PATH.is_file()
parquet_file = pq.ParquetFile(SOURCE_PATH)
source_dataset = ds.dataset(SOURCE_PATH, format="parquet")
selection_table = source_dataset.to_table(
    columns=["date", "store_nbr", "item_nbr"],
    filter=ds.field("date").isin(
        [TRAINING_ORIGINS[0].to_pydatetime(), VALIDATION_ORIGIN.to_pydatetime()]
    ) & ds.field("store_nbr").isin(STORES),
)
selection_frame = selection_table.to_pandas()
selection_frame["date"] = pd.to_datetime(selection_frame["date"])
selected_by_store = {}
selection_evidence = []
for store_nbr in STORES:
    earliest_items = set(selection_frame.loc[
        (selection_frame["store_nbr"] == store_nbr)
        & (selection_frame["date"] == TRAINING_ORIGINS[0]), "item_nbr"
    ])
    fold_origin_items = set(selection_frame.loc[
        (selection_frame["store_nbr"] == store_nbr)
        & (selection_frame["date"] == VALIDATION_ORIGIN), "item_nbr"
    ])
    eligible_items = sorted(earliest_items & fold_origin_items)
    selected_items = tuple(int(value) for value in eligible_items[:ITEMS_PER_STORE])
    assert len(selected_items) == ITEMS_PER_STORE
    selected_by_store[store_nbr] = selected_items
    selection_evidence.append({
        "store_nbr": store_nbr,
        "earliest_origin_items": len(earliest_items),
        "fold_origin_items": len(fold_origin_items),
        "intersection_items": len(eligible_items),
        "selected_items": len(selected_items),
    })

entity_filter = None
for store_nbr, item_nbrs in selected_by_store.items():
    clause = (ds.field("store_nbr") == store_nbr) & ds.field("item_nbr").isin(item_nbrs)
    entity_filter = clause if entity_filter is None else entity_filter | clause
bounded_table = source_dataset.to_table(
    columns=list(SOURCE_READ_COLUMNS),
    filter=(ds.field("date") >= READ_START.to_pydatetime())
    & (ds.field("date") <= READ_END.to_pydatetime())
    & entity_filter,
)
source_frame = bounded_table.to_pandas()
source_frame["date"] = pd.to_datetime(source_frame["date"])
source_frame = source_frame.sort_values(["date", "store_nbr", "item_nbr"]).reset_index(drop=True)

assert source_frame["date"].min() == READ_START
assert source_frame["date"].max() == READ_END
assert not source_frame.duplicated(["date", "store_nbr", "item_nbr"]).any()
assert len(selected_by_store) == 5 and sum(map(len, selected_by_store.values())) == 500

display(pd.DataFrame(selection_evidence))

,store_nbr,earliest_origin_items,fold_origin_items,intersection_items,selected_items
0,1,1475,2092,1143,100
1,2,2458,2240,1839,100
2,3,2773,2731,2353,100
3,4,2356,2171,1765,100
4,5,1912,1936,1349,100


## 3. Source-data scope

The source-scope table distinguishes Parquet metadata from the bounded rows actually projected into memory. `Parquet metadata rows` is the total row count stored in file metadata; it does not mean all source rows were materialized.

In [3]:
display(pd.DataFrame({
    "source_scope_property": [
        "Parquet metadata rows",
        "Parquet row groups",
        "bounded projected rows",
        "bounded date minimum",
        "bounded date maximum",
        "projected source columns",
    ],
    "value": [
        parquet_file.metadata.num_rows,
        parquet_file.metadata.num_row_groups,
        len(source_frame),
        source_frame["date"].min().date(),
        source_frame["date"].max().date(),
        len(SOURCE_READ_COLUMNS),
    ],
}))

,source_scope_property,value
0,Parquet metadata rows,125497040
1,Parquet row groups,502
2,bounded projected rows,72838
3,bounded date minimum,2017-01-31
4,bounded date maximum,2017-07-16
5,projected source columns,20


## 4. Training origins and final dataset sizes

The approved reusable feature builder generates static, historical, calendar, transaction, oil, and conditional future-known fields. Planned promotion and holiday assumptions remain disabled.

In the origin table, `forecast_origin` is the cutoff date known to the model, `rows` is the number of supervised examples generated for that origin, `target_start` is origin plus 1 day, and `target_end` is origin plus 16 days.

In [4]:
training_parts = []
origin_summaries = []
for training_origin in TRAINING_ORIGINS:
    origin_frame = build_feature_rows_for_origin(
        source_frame,
        forecast_origin=training_origin,
        max_items_per_store=None,
        allow_assumed_future_promotion=False,
        allow_assumed_future_holidays=False,
    )
    assert not origin_frame.empty
    assert origin_frame["forecast_date"].max() <= VALIDATION_ORIGIN
    training_parts.append(origin_frame)
    origin_summaries.append({
        "forecast_origin": training_origin.date(),
        "rows": len(origin_frame),
        "target_start": origin_frame["forecast_date"].min().date(),
        "target_end": origin_frame["forecast_date"].max().date(),
    })
training_frame = pd.concat(training_parts, ignore_index=True)
validation_frame = build_feature_rows_for_origin(
    source_frame,
    forecast_origin=VALIDATION_ORIGIN,
    max_items_per_store=None,
    allow_assumed_future_promotion=False,
    allow_assumed_future_holidays=False,
)

assert training_frame["forecast_date"].max() <= VALIDATION_ORIGIN
assert validation_frame["forecast_date"].min() == VALIDATION_START
assert validation_frame["forecast_date"].max() == VALIDATION_END
assert not training_frame.duplicated(["forecast_origin", "forecast_date", "store_nbr", "item_nbr"]).any()
assert not validation_frame.duplicated(["forecast_origin", "forecast_date", "store_nbr", "item_nbr"]).any()

display(pd.DataFrame(origin_summaries))

,forecast_origin,rows,target_start,target_end
0,2017-02-28,7062,2017-03-01,2017-03-16
1,2017-03-14,7062,2017-03-15,2017-03-30
2,2017-03-28,7048,2017-03-29,2017-04-13
3,2017-04-11,6852,2017-04-12,2017-04-27
4,2017-04-25,6943,2017-04-26,2017-05-11
5,2017-05-09,6812,2017-05-10,2017-05-25
6,2017-05-23,6934,2017-05-24,2017-06-08
7,2017-06-06,7005,2017-06-07,2017-06-22
8,2017-06-14,7076,2017-06-15,2017-06-30


### Final training and Fold 8 validation sizes

This summary shows the final example counts, target-date boundaries, stores, and unique items supplied to the bounded training and validation stages.

In [5]:
display(pd.DataFrame({
    "dataset": ["training", "Fold 8 validation"],
    "rows": [len(training_frame), len(validation_frame)],
    "target_start": [training_frame["forecast_date"].min().date(), validation_frame["forecast_date"].min().date()],
    "target_end": [training_frame["forecast_date"].max().date(), validation_frame["forecast_date"].max().date()],
    "stores": [training_frame["store_nbr"].nunique(), validation_frame["store_nbr"].nunique()],
    "unique_items": [training_frame["item_nbr"].nunique(), validation_frame["item_nbr"].nunique()],
}))

,dataset,rows,target_start,target_end,stores,unique_items
0,training,62794,2017-03-01,2017-06-30,5,187
1,Fold 8 validation,6976,2017-07-01,2017-07-16,5,186


### Signed target evidence

The bounded training data contains 10 negative targets and Fold 8 validation contains none. Source targets are preserved without silent clipping or transformation; this table documents that policy rather than resolving later metric treatment.

In [6]:
negative_training_targets = int((training_frame["unit_sales"] < 0).sum())
negative_validation_targets = int((validation_frame["unit_sales"] < 0).sum())
display(pd.DataFrame({
    "dataset": ["bounded training", "Fold 8 validation"],
    "rows": [len(training_frame), len(validation_frame)],
    "negative_target_rows": [negative_training_targets, negative_validation_targets],
    "minimum_target": [training_frame["unit_sales"].min(), validation_frame["unit_sales"].min()],
}))

,dataset,rows,negative_target_rows,minimum_target
0,bounded training,62794,10,-5.00
1,Fold 8 validation,6976,0,0.65


## 5. Temporal and leakage validation

The feature-time boundary table states the latest source time permitted for each historical feature group. Negative offsets mean strictly pre-origin evidence; zero means information available at the forecast origin.

A counterfactual check also mutates post-origin sales, transactions, and oil values and requires the rebuilt Fold 8 inference features to remain unchanged.

In [7]:
future_mutated_source = source_frame.copy()
future_mask = future_mutated_source["date"] > VALIDATION_ORIGIN
future_mutated_source.loc[future_mask, "unit_sales"] += 1_000_000.0
future_mutated_source.loc[future_mask, "transactions"] = 1_000_000
future_mutated_source.loc[future_mask, "dcoilwtico"] = 1_000_000.0
mutated_validation_features = build_feature_rows_for_origin(
    future_mutated_source,
    forecast_origin=VALIDATION_ORIGIN,
    max_items_per_store=None,
    allow_assumed_future_promotion=False,
    allow_assumed_future_holidays=False,
)
pd.testing.assert_frame_equal(
    validation_frame.loc[:, list(INFERENCE_OUTPUT_COLUMNS)],
    mutated_validation_features.loc[:, list(INFERENCE_OUTPUT_COLUMNS)],
)

historical_source_boundaries = pd.DataFrame({
    "feature_group": ["sales lags", "sales rolling windows", "transactions", "oil movements", "static attributes"],
    "latest_allowed_source_time": ["origin - 1 day or earlier", "origin - 1 day", "origin", "origin", "origin"],
    "maximum_offset_days_from_origin": [-1, -1, 0, 0, 0],
})
leakage_checks = {
    "training_label_cutoff": training_frame["forecast_date"].le(VALIDATION_ORIGIN).all(),
    "training_origins_precede_labels": training_frame["forecast_origin"].lt(training_frame["forecast_date"]).all(),
    "historical_sources_not_future": historical_source_boundaries["maximum_offset_days_from_origin"].le(0).all(),
    "future_actual_mutation_does_not_change_features": True,
    "promotion_unknown_without_as_of_evidence": training_frame["onpromotion"].isna().all() and validation_frame["onpromotion"].isna().all(),
    "holiday_fields_unknown_without_as_of_evidence": all(training_frame[column].isna().all() and validation_frame[column].isna().all() for column in HOLIDAY_FEATURE_COLUMNS),
    "validation_dates_exact_fold_8": validation_frame["forecast_date"].between(VALIDATION_START, VALIDATION_END).all(),
    "final_holdout_not_materialized": source_frame["date"].max() < pd.Timestamp(FINAL_HOLDOUT.holdout_start),
}
assert all(leakage_checks.values())
display(historical_source_boundaries)

,feature_group,latest_allowed_source_time,maximum_offset_days_from_origin
0,sales lags,origin - 1 day or earlier,-1
1,sales rolling windows,origin - 1 day,-1
2,transactions,origin,0
3,oil movements,origin,0
4,static attributes,origin,0


### Temporal-control assertion results

This concise table reports the required label, source-time, mutation, availability, Fold 8, and holdout assertions. `True` means that specific leakage or temporal-control assertion passed.

In [8]:
display(pd.DataFrame(
    leakage_checks.items(),
    columns=["validation_check", "passed"],
))

,validation_check,passed
0,training_label_cutoff,True
1,training_origins_precede_labels,True
2,historical_sources_not_future,True
3,future_actual_mutation_does_not_change_features,True
4,promotion_unknown_without_as_of_evidence,True
5,holiday_fields_unknown_without_as_of_evidence,True
6,validation_dates_exact_fold_8,True
7,final_holdout_not_materialized,True


## 6. Final model matrix

The model-matrix table reports training rows, validation rows, fitted feature count, categorical feature count, and the number of all-null training features excluded. The explicit lists show which categorical fields were used and which unavailable fields were removed.

All-null features contain no usable bounded training information. Partially missing features remain missing for LightGBM native handling; no blanket zero filling or target encoding is applied.

In [9]:
candidate_feature_columns = ("forecast_horizon", *MODEL_FEATURE_COLUMNS)
all_null_training_features = tuple(
    column for column in candidate_feature_columns if training_frame[column].isna().all()
)
feature_columns = tuple(
    column for column in candidate_feature_columns if column not in all_null_training_features
)
categorical_candidates = (
    "store_nbr", "item_nbr", "family", "class",
    "city", "state", "store_type", "cluster",
    "holiday_type", "holiday_locale",
)
categorical_features = tuple(
    column for column in categorical_candidates if column in feature_columns
)
X_train = training_frame.loc[:, feature_columns].copy()
X_validation = validation_frame.loc[:, feature_columns].copy()
for column in categorical_features:
    training_categories = pd.Index(X_train[column].dropna().unique()).sort_values()
    X_train[column] = pd.Categorical(X_train[column], categories=training_categories)
    X_validation[column] = pd.Categorical(X_validation[column], categories=training_categories)
for frame in (X_train, X_validation):
    for column in frame.columns:
        if str(frame[column].dtype) == "boolean" or frame[column].dtype == bool:
            frame[column] = frame[column].astype(float)

assert "forecast_horizon" in feature_columns
assert "unit_sales" not in feature_columns
assert not any(str(dtype) in {"object", "string"} for dtype in X_train.dtypes)
assert list(X_train.columns) == list(X_validation.columns)

display(pd.DataFrame({
    "model_matrix_property": ["training rows", "validation rows", "feature count", "categorical feature count", "all-null features excluded"],
    "value": [len(X_train), len(X_validation), len(feature_columns), len(categorical_features), len(all_null_training_features)],
}))
print(f"Categorical features: {categorical_features}")
print(f"All-null bounded features excluded: {all_null_training_features}")

,model_matrix_property,value
0,training rows,62794
1,validation rows,6976
2,feature count,34
3,categorical feature count,8
4,all-null features excluded,6


Categorical features: ('store_nbr', 'item_nbr', 'family', 'class', 'city', 'state', 'store_type', 'cluster')
All-null bounded features excluded: ('onpromotion', 'is_holiday', 'holiday_type', 'holiday_locale', 'holiday_transferred', 'holiday_event_count')


## 7. LightGBM configuration and bounded fit

One global LightGBM regression booster is fitted across all bounded store-item training examples. The configuration table records the complete reproducible first-pass setup.

These are controlled exploratory parameters, not tuned or final parameters. The experiment uses 150 boosting rounds, no early stopping, no parameter search, and four CPU threads.

In [10]:
LIGHTGBM_PARAMETERS = {
    "objective": "regression",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.9,
    "seed": 42,
    "num_threads": 4,
    "verbosity": -1,
    "deterministic": True,
    "force_col_wise": True,
}
NUM_BOOST_ROUND = 150
model_configuration = {**LIGHTGBM_PARAMETERS, "num_boost_round": NUM_BOOST_ROUND}
display(pd.DataFrame(
    model_configuration.items(),
    columns=["model_configuration_parameter", "value"],
))

train_dataset = lgb.Dataset(
    X_train,
    label=training_frame["unit_sales"],
    categorical_feature=list(categorical_features),
    free_raw_data=False,
)
training_started = perf_counter()
global_model = lgb.train(
    LIGHTGBM_PARAMETERS,
    train_dataset,
    num_boost_round=NUM_BOOST_ROUND,
)
training_duration_seconds = perf_counter() - training_started

display(pd.DataFrame({
    "model_training_property": [
        "model type",
        "global models trained",
        "boosting rounds",
        "training duration seconds",
    ],
    "value": [
        type(global_model).__name__,
        1,
        global_model.current_iteration(),
        round(training_duration_seconds, 4),
    ],
}))

,model_configuration_parameter,value
0,objective,regression
1,learning_rate,0.05
2,num_leaves,31
3,min_data_in_leaf,20
4,feature_fraction,0.9
5,seed,42
6,num_threads,4
7,verbosity,-1
8,deterministic,True
9,force_col_wise,True


,model_training_property,value
0,model type,Booster
1,global models trained,1
2,boosting rounds,150
3,training duration seconds,0.4519


## 8. Prediction validation

The single global model produces one direct prediction for every Fold 8 validation row without recursive feedback. Audit keys, horizon coverage, forecast-date arithmetic, uniqueness, finiteness, Fold 8 scope, and holdout separation are checked below.

`True` means that specific prediction-validation assertion passed.

In [11]:
validation_predictions = global_model.predict(
    X_validation, num_iteration=global_model.current_iteration()
)
prediction_table = validation_frame[[
    "forecast_origin", "forecast_date", "forecast_horizon",
    "store_nbr", "item_nbr", "unit_sales",
]].rename(columns={"unit_sales": "actual_unit_sales"})
prediction_table["prediction"] = validation_predictions
prediction_key = ["forecast_origin", "forecast_date", "store_nbr", "item_nbr"]
prediction_checks = {
    "row_count_matches_validation": len(prediction_table) == len(validation_frame),
    "horizons_within_1_16": prediction_table["forecast_horizon"].isin(FORECAST_HORIZONS).all(),
    "all_horizons_observed": tuple(sorted(prediction_table["forecast_horizon"].unique())) == tuple(FORECAST_HORIZONS),
    "forecast_date_equation": (prediction_table["forecast_date"] == prediction_table["forecast_origin"] + pd.to_timedelta(prediction_table["forecast_horizon"], unit="D")).all(),
    "no_duplicate_prediction_keys": not prediction_table.duplicated(prediction_key).any(),
    "numeric_finite_predictions": np.isfinite(prediction_table["prediction"]).all(),
    "fold_8_only": prediction_table["forecast_origin"].eq(VALIDATION_ORIGIN).all(),
    "holdout_untouched": prediction_table["forecast_date"].max() < pd.Timestamp(FINAL_HOLDOUT.holdout_start),
}
for row in prediction_table.itertuples(index=False):
    validate_forecast_date_horizon(row.forecast_origin.date(), row.forecast_date.date(), int(row.forecast_horizon))
assert all(prediction_checks.values())
display(pd.DataFrame(prediction_checks.items(), columns=["prediction_validation_check", "passed"]))

,prediction_validation_check,passed
0,row_count_matches_validation,True
1,horizons_within_1_16,True
2,all_horizons_observed,True
3,forecast_date_equation,True
4,no_duplicate_prediction_keys,True
5,numeric_finite_predictions,True
6,fold_8_only,True
7,holdout_untouched,True


## 9. Exploratory evaluation metrics

MAE and RMSE are bounded Fold 8 diagnostics only. They are not final SCRUM-17 model-selection evidence and do not establish a selected model or research conclusion.

In [12]:
validation_error = prediction_table["actual_unit_sales"].to_numpy() - prediction_table["prediction"].to_numpy()
exploratory_mae = float(np.mean(np.abs(validation_error)))
exploratory_rmse = float(np.sqrt(np.mean(np.square(validation_error))))
display(pd.DataFrame({
    "evaluation_scope": ["Fold 8 exploratory; not final SCRUM-17 evidence"],
    "validation_rows": [len(prediction_table)],
    "MAE": [exploratory_mae],
    "RMSE": [exploratory_rmse],
}).round(4))

,evaluation_scope,validation_rows,MAE,RMSE
0,Fold 8 exploratory; not final SCRUM-17 evidence,6976,3.3496,6.0168


MAE ≈ 3.35 means predictions differed from actual unit sales by about 3.35 units on average in this bounded Fold 8 experiment. RMSE ≈ 6.02 penalizes larger errors more strongly. These values must not be treated as final eight-fold model-selection evidence.

## 10. Actual-versus-prediction sample

This deterministically sorted 10-row preview is a human-readable sample only. Metrics use all 6,976 Fold 8 validation predictions.

In [13]:
sample_predictions = prediction_table.sort_values(
    ["forecast_date", "store_nbr", "item_nbr"]
).head(10)
display(sample_predictions.reset_index(drop=True))

,forecast_origin,forecast_date,forecast_horizon,store_nbr,item_nbr,actual_unit_sales,prediction
0,2017-06-30,2017-07-01,1,1,103665,11.0,4.022297
1,2017-06-30,2017-07-01,1,1,105574,2.0,7.008065
2,2017-06-30,2017-07-01,1,1,105575,3.0,16.073920
3,2017-06-30,2017-07-01,1,1,105577,4.0,3.676171
4,2017-06-30,2017-07-01,1,1,106716,5.0,3.522618
5,2017-06-30,2017-07-01,1,1,108698,3.0,4.533324
6,2017-06-30,2017-07-01,1,1,111223,6.0,8.191513
7,2017-06-30,2017-07-01,1,1,112830,1.0,2.654791
8,2017-06-30,2017-07-01,1,1,114800,5.0,4.754032
9,2017-06-30,2017-07-01,1,1,115267,4.0,4.508573


## 11. Feature importance

`feature` is the model input; `gain` is total LightGBM improvement attributed to splits using that feature; `split_count` is the number of tree splits using it; and `gain_pct` is its percentage of total model gain.

The table retains the top 15 bounded-experiment features. Feature importance indicates model usage, not causal importance.

In [14]:
feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "gain": global_model.feature_importance(importance_type="gain"),
    "split_count": global_model.feature_importance(importance_type="split"),
}).sort_values(["gain", "feature"], ascending=[False, True]).reset_index(drop=True)
total_gain = feature_importance["gain"].sum()
feature_importance["gain_pct"] = np.where(
    total_gain > 0, 100 * feature_importance["gain"] / total_gain, 0.0
)
display(feature_importance.head(15).round({"gain": 2, "gain_pct": 2}))

,feature,gain,split_count,gain_pct
0,sales_rolling_mean_7,14499212.79,241,47.41
1,item_nbr,4765855.99,1094,15.58
2,sales_rolling_mean_14,1946391.29,154,6.36
3,day_of_week,1195342.85,390,3.91
4,day_of_month,1177585.66,410,3.85
5,sales_lag_1,912006.92,189,2.98
6,sales_lag_28,832994.06,191,2.72
7,week_of_year,668933.79,184,2.19
8,transactions_at_origin,482263.51,100,1.58
9,sales_rolling_mean_28,436147.78,121,1.43


## 12. Comparison boundary and limitations

Notebook 10 used Store 1 and 50 items selected from Fold 8 origin rows only. This notebook uses Stores 1–5 and 100 items per store from an intersection observed at two pre-validation origins. Because the populations differ, their saved errors are not a fair head-to-head comparison; later baseline review must use identical prediction keys.

This experiment covers five stores, a deterministic 500-pair selection, nine recent training origins, and Fold 8 only. It is not full-population training, approved eight-fold real-data backtesting, final-holdout evidence, model comparison, final model selection, production readiness, or cloud readiness.

## 13. Concise completion summary

The final table consolidates high-level experiment status without repeating the detailed temporal and prediction assertions above. `True` means the stated bounded completion condition holds.

In [15]:
completion_checks = {
    "one_global_lightgbm_model": type(global_model).__name__ == "Booster",
    "forecast_horizon_used_as_feature": "forecast_horizon" in feature_columns,
    "no_separate_store_item_or_horizon_models": True,
    "fold_8_validation_completed": len(prediction_table) == len(validation_frame),
    "temporal_and_leakage_controls_passed": all(leakage_checks.values()),
    "prediction_validation_passed": all(prediction_checks.values()),
    "bounded_source_scope": len(source_frame) < parquet_file.metadata.num_rows,
    "final_holdout_untouched": source_frame["date"].max() < pd.Timestamp(FINAL_HOLDOUT.holdout_start),
    "no_hyperparameter_search": True,
    "not_full_eight_fold_production_evaluation": True,
}
assert all(completion_checks.values())
display(pd.DataFrame(
    completion_checks.items(),
    columns=["experiment_completion_check", "passed"],
))

,experiment_completion_check,passed
0,one_global_lightgbm_model,True
1,forecast_horizon_used_as_feature,True
2,no_separate_store_item_or_horizon_models,True
3,fold_8_validation_completed,True
4,temporal_and_leakage_controls_passed,True
5,prediction_validation_passed,True
6,bounded_source_scope,True
7,final_holdout_untouched,True
8,no_hyperparameter_search,True
9,not_full_eight_fold_production_evaluation,True


This notebook demonstrates a bounded first global LightGBM forecasting experiment.

It does not constitute:

- full-data training;
- approved eight-fold real-data backtesting;
- model comparison;
- final model selection; or
- final holdout scoring.